# kwargs-pass-through-recipe — faded example 2: Store kwargs in Recipe — fill in the Recipe construction

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `kwargs-pass-through-recipe`. Running the beacon reports progress on the `Backprop: Kwargs pass-through` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Kwargs pass-through` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kwargs-pass-through-recipe`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kwargs-pass-through-recipe"
DD_SUBTOPIC = "Backprop: Kwargs pass-through"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A `Recipe` stores the function, raw positional arguments, keyword arguments, and parent tensors. Storing `dict(kwargs)` (a copy) rather than `kwargs` directly prevents the stored Recipe from being mutated if the caller modifies the dict after the call. The backward function will later splat `**recipe.kwargs` to replay the forward op.

## Faded exercise 2

The forward call is complete and `out_raw` is computed. Fill in the Recipe construction so it stores all four fields including a copy of `kwargs`.

**Fill in:** Build out.recipe as a Recipe with func=fwd_fn, args=raw_args, kwargs=a copy of kwargs, and parents from the MiniTensor arguments.

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Any, Callable, Optional

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array):
        self.array = array
        self.recipe: Optional[Recipe] = None

def wrap_forward_fn(fwd_fn: Callable) -> Callable:
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_raw = fwd_fn(*raw_args, **kwargs)
        parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
        out = MiniTensor(out_raw)
        out.recipe = Recipe(fwd_fn, raw_args, dict(kwargs), parents)
        return out
    return tensor_func


def _test():
    import numpy as np

    wrapped_mean = wrap_forward_fn(np.mean)
    x = MiniTensor(np.arange(6.0).reshape(2, 3))

    out = wrapped_mean(x, axis=0)
    assert out.recipe is not None
    assert out.recipe.func is np.mean
    assert out.recipe.kwargs == {'axis': 0}
    assert isinstance(out.recipe.args, tuple)
    assert 0 in out.recipe.parents
    assert out.recipe.parents[0] is x


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
from dataclasses import dataclass
from typing import Any, Callable, Optional

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array):
        self.array = array
        self.recipe: Optional[Recipe] = None

def wrap_forward_fn(fwd_fn: Callable) -> Callable:
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_raw = fwd_fn(*raw_args, **kwargs)
        parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
        out = MiniTensor(out_raw)
        out.recipe = Recipe(fwd_fn, raw_args, dict(kwargs), parents)
        return out
    return tensor_func
```
</details>